In [2]:
import os, glob, ast
import pandas as pd
from collections import Counter

import dash
from dash import Dash, html, dcc, Input, Output, ALL, no_update

ARTISTS_DIR = "artists_csv"      # folder with one CSV per artist
TOP_N_ARTISTS = 20               # how many artist KPI cards to show
SONG_SORT_BY = "song_popularity_2025"  # fallback to 'popularity' if absent

In [3]:
def human_count(n):
    try:
        n = float(n)
    except Exception:
        return None
    units = ["", "K", "M", "B", "T"]
    k = 0
    while abs(n) >= 1000 and k < len(units) - 1:
        n /= 1000.0; k += 1
    return f"{n:.1f}{units[k]}"

In [4]:
def read_artist_csv(path):
    """Read one artist CSV; return (df, artist_name)."""
    df = pd.read_csv(path)
    # normalize columns we’ll use
    cols = {c: c.strip() for c in df.columns}
    df.rename(columns=cols, inplace=True)
    artist_name = df["artist_name"].iloc[0]
    return df, artist_name


In [5]:
def top_genres(genres_series, k=3):
    # genres may be comma string or list-like string; normalize
    def parse(val):
        if pd.isna(val): return []
        if isinstance(val, list): return val
        s = str(val)
        # try list literal
        try:
            x = ast.literal_eval(s)
            if isinstance(x, list): return [str(g) for g in x]
        except Exception:
            pass
        return [g.strip() for g in s.split(",") if g.strip()]
    all_g = []
    for v in genres_series.fillna(""):
        all_g.extend(parse(v))
    if not all_g: return []
    c = Counter(all_g)
    return [g for g,_ in c.most_common(k)]

In [ ]:
def summarize_artist_file(path):
    """Return one row of summary for an artist (for the KPI card row)."""
    df, artist_name = read_artist_csv(path)
    n_songs = len(df)

    # followers / artist popularity / image (take max/first non-null)
    followers = df["followers"].dropna().max() if "followers" in df.columns else None
    a_pop     = df["popularity_artist_2025"].dropna().max() if "popularity_artist_2025" in df.columns else None
    img       = df["image_url_artist"].dropna().iloc[0] if "image_url_artist" in df.columns and df["image_url_artist"].notna().any() else None

    # track popularity across this artist’s Top-40 slice
    pop_col = SONG_SORT_BY if SONG_SORT_BY in df.columns else ("popularity" if "popularity" in df.columns else None)
    avg_pop = df[pop_col].mean() if pop_col else None

    # explicit %
    pct_explicit = 100.0 * df.get("explicit", pd.Series([False]*n_songs)).fillna(False).astype(bool).mean()

    # audio profile averages (if present in your Kaggle slice)
    mean_dance  = df.get("danceability", pd.Series([None]*n_songs)).mean()
    mean_energy = df.get("energy", pd.Series([None]*n_songs)).mean()
    mean_val    = df.get("valence", pd.Series([None]*n_songs)).mean()

    # top genres (across tracks for this artist)
    g3 = top_genres(df.get("genre", pd.Series([""]*n_songs)))

    return {
        "artist_name": artist_name,
        "file_path": path,
        "songs_in_slice": n_songs,
        "followers": followers,
        "followers_h": human_count(followers) if followers is not None else None,
        "artist_popularity": a_pop,
        "artist_image_url": img,
        "avg_track_popularity": round(avg_pop, 1) if pd.notna(avg_pop) else None,
        "pct_explicit": round(pct_explicit, 1),
        "mean_dance": None if pd.isna(mean_dance) else round(float(mean_dance), 2),
        "mean_energy": None if pd.isna(mean_energy) else round(float(mean_energy), 2),
        "mean_valence": None if pd.isna(mean_val) else round(float(mean_val), 2),
        "genres_top3": g3,
    }
